# ChubbChurns - AI Explainability Demo

This notebook demonstrates how to use SHAP and LIME to explain customer churn predictions.

## What You'll Learn:
1. How to generate and preprocess customer churn data
2. How to train a churn prediction model
3. How to use SHAP for global and local explanations
4. How to use LIME for instance-level explanations
5. How to interpret and visualize model predictions

## 1. Setup and Data Generation

In [ ]:
# Import required libraries
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_processing import generate_sample_churn_data, preprocess_data
from src.model import ChurnPredictor
from src.explainability import SHAPExplainer, LIMEExplainer

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")

In [ ]:
# Generate synthetic customer churn data
print("Generating sample customer data...")
data = generate_sample_churn_data(n_samples=1000, random_state=42)

print(f"\nDataset shape: {data.shape}")
print(f"\nFeatures: {data.columns.tolist()}")
print(f"\nChurn distribution:\n{data['churn'].value_counts()}")
print(f"\nChurn rate: {data['churn'].mean():.2%}")

# Display sample records
data.head(10)

## 2. Data Preprocessing and Model Training

In [ ]:
# Preprocess the data
print("Preprocessing data...")
X_train, X_test, y_train, y_test, scaler, feature_names = preprocess_data(data)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")
print(f"Features: {feature_names}")

In [ ]:
# Train a Random Forest model
print("Training Random Forest model...")
model = ChurnPredictor(model_type='random_forest', random_state=42)
model.train(X_train, y_train)

# Evaluate the model
metrics = model.evaluate(X_test, y_test)
print("\nModel Performance:")
for metric_name, value in metrics.items():
    print(f"  {metric_name}: {value:.4f}")

In [ ]:
# Display feature importance from the model
feature_importance = model.get_feature_importance()
print("\nFeature Importance:")
print(feature_importance)

# Plot feature importance
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance from Random Forest')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 3. SHAP Explanations

SHAP (SHapley Additive exPlanations) provides a unified approach to explain model predictions using game theory concepts.

In [ ]:
# Create SHAP explainer
print("Creating SHAP explainer...")
shap_explainer = SHAPExplainer(model, feature_names=feature_names)
shap_explainer.create_explainer(X_train.sample(100, random_state=42), algorithm='tree')

# Generate SHAP values for test set
print("Generating SHAP values...")
shap_values = shap_explainer.explain(X_test[:100])
print(f"SHAP values shape: {shap_values.shape}")

In [ ]:
# SHAP Summary Plot - Global Feature Importance
print("Creating SHAP summary plot...")
shap_explainer.plot_summary(X_test[:100], save_path='../visualizations/shap_summary.png')

# Display the plot
import shap
shap.summary_plot(shap_values, X_test[:100], feature_names=feature_names)

In [ ]:
# SHAP Waterfall Plot - Single Instance Explanation
print("Creating SHAP waterfall plot for first test instance...")
instance_idx = 0

print(f"\nCustomer {instance_idx} features:")
print(X_test.iloc[instance_idx])
print(f"\nActual churn: {y_test.iloc[instance_idx]}")
print(f"Predicted churn probability: {model.predict_proba(X_test.iloc[[instance_idx]])[0, 1]:.4f}")

shap_explainer.plot_waterfall(X_test, instance_index=instance_idx, 
                              save_path='../visualizations/shap_waterfall.png')

## 4. LIME Explanations

LIME (Local Interpretable Model-agnostic Explanations) explains individual predictions by approximating the model locally with an interpretable model.

In [ ]:
# Create LIME explainer
print("Creating LIME explainer...")
lime_explainer = LIMEExplainer(model, feature_names=feature_names)
lime_explainer.create_explainer(X_train)

print("LIME explainer created successfully!")

In [ ]:
# Explain a single instance with LIME
instance_idx = 0
print(f"Explaining customer {instance_idx} with LIME...")

explanation = lime_explainer.explain_instance(X_test.iloc[instance_idx], num_features=10)

# Display feature contributions
contributions = lime_explainer.get_feature_contributions(explanation)
print("\nFeature Contributions:")
print(contributions)

# Plot explanation
lime_explainer.plot_explanation(explanation, save_path='../visualizations/lime_explanation.png')

# Show the explanation
explanation.show_in_notebook()

## 5. Comparing Multiple Instances

In [ ]:
# Analyze multiple customer instances
print("Analyzing multiple customer instances...\n")

for i in range(3):
    print(f"\n{'='*60}")
    print(f"Customer {i}:")
    print(f"{'='*60}")
    
    # Get prediction
    proba = model.predict_proba(X_test.iloc[[i]])[0, 1]
    actual = y_test.iloc[i]
    
    print(f"Actual churn: {actual}")
    print(f"Predicted churn probability: {proba:.4f}")
    print(f"Prediction: {'CHURN' if proba >= 0.5 else 'NO CHURN'}")
    
    # LIME explanation
    explanation = lime_explainer.explain_instance(X_test.iloc[i], num_features=5)
    contributions = lime_explainer.get_feature_contributions(explanation)
    
    print(f"\nTop 5 Feature Contributions:")
    print(contributions.head())

## 6. Business Insights

Based on the explainability analysis, we can derive actionable business insights:

In [ ]:
# Calculate average SHAP values for global feature importance
mean_abs_shap = np.abs(shap_values).mean(axis=0)
feature_importance_shap = pd.DataFrame({
    'feature': feature_names,
    'mean_abs_shap': mean_abs_shap
}).sort_values('mean_abs_shap', ascending=False)

print("Global Feature Importance (SHAP):")
print(feature_importance_shap)

# Visualize
plt.figure(figsize=(10, 6))
plt.barh(feature_importance_shap['feature'], feature_importance_shap['mean_abs_shap'])
plt.xlabel('Mean Absolute SHAP Value')
plt.title('Global Feature Importance using SHAP')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("KEY INSIGHTS:")
print("="*60)
print("1. The most important features for churn prediction are:")
for idx, row in feature_importance_shap.head(3).iterrows():
    print(f"   - {row['feature']}")
print("\n2. Focus retention efforts on customers with:")
print("   - Low tenure (new customers)")
print("   - High customer service call frequency")
print("   - Higher monthly charges")
print("\n3. Consider improving:")
print("   - Customer onboarding experience")
print("   - Customer support quality")
print("   - Value proposition for high-paying customers")

## Summary

In this notebook, we demonstrated:
1. ✅ Training a customer churn prediction model
2. ✅ Using SHAP for global and local explanations
3. ✅ Using LIME for instance-level explanations
4. ✅ Deriving actionable business insights from model explanations

### Next Steps:
- Try different model types (Gradient Boosting, Logistic Regression)
- Experiment with different SHAP explainer types
- Analyze specific customer segments
- Build a dashboard for real-time churn prediction and explanation